In [ ]:
# ========== 第 1 天任务：抓取电商页 → 用 GPT 抽取目标商品列表 ==========
# 练习目标：把「网页抓取 + Chat Completions」串起来，按商品名筛出带价格的列表（Markdown 输出）
# 怎么跑：准备好 .env（OPENAI_API_KEY）后，从上到下运行；末尾可改 URL / 商品名再试

# 导入标准库 os：读环境变量（本格主要靠 OpenAI 默认读 OPENAI_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮展示模型返回的 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI
# 从 bs4 导入 BeautifulSoup：解析 HTML，去掉无关标签后抽出纯文本
from bs4 import BeautifulSoup
# 导入 requests：用 HTTP GET 下载目标网页
import requests


# 浏览器风格 User-Agent：很多网站会拒无头爬虫；字符串保持原样（影响请求是否成功）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# 加载 .env 到环境变量
load_dotenv()
# 创建默认 OpenAI 客户端（密钥来自环境变量 OPENAI_API_KEY）
openai = OpenAI()



# system prompt：角色 + 抽取规则 + 输出格式；必须保留英文原文，改译会改变模型行为
system_prompt = """You are an AI product extraction assistant.

Your task is to analyze raw scraped e-commerce website content and extract structured product listings that match a specified product name provided by the user.

You will be given:

1. The scraped website content (HTML or text)
2. A target product name

Instructions:

* Identify all products in the content that match or are closely related to the target product name.

* Ignore text that may be navigation related, including menus, headers, footers, category links, breadcrumbs, filters, login sections, advertisements, or pagination elements.

* Focus only on actual product listing information.

* Extract at least 10 relevant product listings where available.

* For each product extract:

  * Product Name
  * Product Description
  * Product Price

Data Handling Rules:

* Ignore listings that do not contain a visible price.
* Normalize all prices into numeric values (remove currency symbols).
* If a price is given as a range, use the higher value.
* Ignore advertisements or unrelated products.

Sorting Rules:
* Sort the final list strictly on price and quality.


Output Format:

Respond in markdown.
Do not wrap the markdown in a code block - respond just with the markdown.

Return the result as a Markdown that includes Product Name Description and Price 

Order in ascending order with the lowest price at the top
Provide a summary after the table of the most recomended as per the price.

* The result must contain at least 10 products where available.
* Prices must be shown as numeric values with currency.

 """
# user prompt 前缀：后面会拼接「网页正文 + 目标商品名」；英文原文保留
user_prompt = """
 Here is the website content and the product name

"""

def fetch_website_contents(url):
    """抓取 url 对应页面，清洗 HTML 后返回「标题 + 正文」截断文本（最多约 2000 字符）。"""
    # GET 目标页；带上 headers 降低被拒概率
    response = requests.get(url, headers=headers)
    # 用 html.parser 解析响应字节为 DOM
    soup = BeautifulSoup(response.content, "html.parser")
    # 取 <title> 文本；没有标题就用占位英文串（字符串保持原样）
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删掉 script/style/img/input 等对摘要无用的节点，减少噪声
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 从 body 抽出可见文本；换行分隔并去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 没有 body 时正文为空串
        text = ""
    # 标题与正文用空行拼接，再截断到 2000 字符，控制发给模型的上下文长度
    return (title + "\n\n" + text)[:2_000]


def messages_for(website, product_name):
    """组装 Chat Completions 所需的 messages：system 定规则，user 放网页内容 + 商品名。"""
    return [
        {"role": "system", "content": system_prompt},
        # user 内容 = 前缀 + 抓取文本 + 目标商品名（拼接方式保持原样）
        {"role": "user", "content": user_prompt + website+ product_name}
    ]


def summarize(url, product_name):
    """抓取网页 → 调 gpt-4.1-mini → 返回模型生成的 Markdown 字符串。"""
    # 先拿到清洗后的网站文本
    website = fetch_website_contents(url)
    # 非流式 Chat Completions：等整段答完再取 content
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website, product_name)
    )
    # 取第一条 choice 的消息正文
    return response.choices[0].message.content

def display_summary(url, product_name):
    """调用 summarize，并在笔记本里用 Markdown 渲染结果。"""
    summary = summarize(url, product_name)
    display(Markdown(summary))    


# 入口示例：抓 amazon.com，目标商品名为 static bikes（URL / 查询串保持原样）
display_summary("https://amazon.com", "static bikes")
